# D: Guidebook RAG (Yasa)

Steps 1-5: load the guidebook, build the FAISS index, recall@3 on `data/test_sentences.csv`,
tune query type and k, list entries that are never retrieved (for Amen to reword).
Runs on CPU; no GPU needed. Retrieval code lives in `src/rag.py`; this notebook only evaluates it.

In [ ]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

%load_ext autoreload
%autoreload 2

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

## D0. Guidebook and test sentences
Until Amen's guidebook and test sentences land, point these at a local draft **outside the repo**
(e.g. on Drive). Never commit a draft as `data/guidebook.md`. Leave as `None` to use the team files.

In [ ]:
DRAFT_GUIDEBOOK = os.environ.get("DEBIAS_DRAFT_GUIDEBOOK")            # e.g. "/content/drive/MyDrive/debias/draft_guidebook.md"
DRAFT_TEST_SENTENCES = os.environ.get("DEBIAS_DRAFT_TEST_SENTENCES")  # same columns as data/test_sentences.csv

if DRAFT_GUIDEBOOK:
    config.GUIDEBOOK_PATH = Path(DRAFT_GUIDEBOOK)  # src/rag.py reads this at call time
TEST_SENTENCES = Path(DRAFT_TEST_SENTENCES) if DRAFT_TEST_SENTENCES else config.TEST_SENTENCES_PATH
print("guidebook:", config.GUIDEBOOK_PATH, "| draft" if DRAFT_GUIDEBOOK else "")
print("test sentences:", TEST_SENTENCES, "| draft" if DRAFT_TEST_SENTENCES else "")

## D1. Load entries and check ids

In [ ]:
import re
from src import rag

entries = rag.load_entries(config.GUIDEBOOK_PATH)
ids = [rag.entry_id(e) for e in entries]
for e in entries:
    print(e.splitlines()[0])
bad = [i for i in ids if not re.fullmatch(r"R\d{2}", i)]
dupes = sorted({i for i in ids if ids.count(i) > 1})
assert not bad and not dupes, f"bad ids: {bad}, duplicate ids: {dupes}"
assert config.ALWAYS_INCLUDED_RULE in ids
print(len(entries), "entries")

## D2. Build the index and try a few queries
The first call downloads `config.EMBEDDING_MODEL` and builds the index; later calls reuse it until the
guidebook file changes. `rag.rebuild_index()` forces a rebuild.

In [ ]:
import time

assert not rag._use_fake(), "sentence-transformers/faiss missing or DEBIAS_FAKE=1: this would test the word-overlap fallback"
t = time.time()
rag.rebuild_index()
rag.retrieve("warmup")
print(f"index built in {time.time() - t:.1f}s")

queries = [  # keep them symmetric: one per side
    "Side A terrorists",
    "Side B terrorists",
    "the regime butchered civilians",
    "Side B forces shelled the hospital",
]
for q in queries:
    print(f"{q!r:45s} -> {rag.retrieve_ids(q, config.RAG_K)}")

## D3. Recall@3 on the test sentences
A sentence is a hit if any of its `expected_rules` (separated by `;`, R01 ignored because it is always
included) is among the top k retrieved. Two query types:
- `flagged`: words from `explain()` joined by spaces, as `src/pipeline.py` does (fake detector for now, so
  this falls back to the whole sentence when nothing is flagged);
- `sentence`: the whole biased sentence.

In [ ]:
import pandas as pd
from src.detector import explain

tests = pd.read_csv(TEST_SENTENCES)
print(len(tests), "test sentences")
if tests.empty:
    print("No test sentences yet (Amen, Person C). Set DRAFT_TEST_SENTENCES in D0 to test on a draft.")


def expected(rules: str) -> set[str]:
    return {r.strip() for r in str(rules).split(";") if r.strip()} - {config.ALWAYS_INCLUDED_RULE}


def make_query(row, mode: str) -> str:
    if mode == "flagged":
        spans = explain(row["biased"], row["conflict"])
        return " ".join(spans) if spans else row["biased"]
    return row["biased"]


def evaluate(df: pd.DataFrame, mode: str, k: int) -> tuple[dict, pd.DataFrame]:
    rows = []
    for _, row in df.iterrows():
        exp = expected(row["expected_rules"])
        if not exp:
            continue
        got = [i for i in rag.retrieve_ids(make_query(row, mode), k) if i != config.ALWAYS_INCLUDED_RULE][:k]
        rows.append({"id": row["id"], "expected": sorted(exp), "retrieved": got, "hit": bool(exp & set(got))})
    res = pd.DataFrame(rows, columns=["id", "expected", "retrieved", "hit"])
    recall = float(res["hit"].mean()) if len(res) else float("nan")
    return {"recall": recall, "n": len(res)}, res


metrics, res = evaluate(tests, "sentence", config.RAG_K)
print(f"recall@{config.RAG_K} (sentence): {metrics['recall']:.3f} on {metrics['n']} sentences")
misses = res[~res["hit"]]
print(misses.explode("expected")["expected"].value_counts().rename("misses per expected rule"))
misses

## D4. Tuning: query type x k
`k = 4` is only worth it if recall improves clearly: the rewriter prompt gets longer with every rule.

In [ ]:
grid = {}
for mode in ("flagged", "sentence"):
    for k in (3, 4):
        m, _ = evaluate(tests, mode, k)
        grid[f"{mode}_k{k}"] = m
        print(f"{mode:9s} k={k}: recall={m['recall']:.3f}  n={m['n']}")

utils.save_result(
    "rag_recall",
    {name: m["recall"] for name, m in grid.items()},
    extra={"n": {name: m["n"] for name, m in grid.items()}, "embedding_model": config.EMBEDDING_MODEL,
           "guidebook": str(config.GUIDEBOOK_PATH), "draft": bool(DRAFT_GUIDEBOOK or DRAFT_TEST_SENTENCES),
           "fake_detector": config.USE_FAKES or not config.DETECTOR_PATH},
)

## D5. Entries never retrieved
Rules that never show up in the top k for any test sentence (either query type). Send the list to Amen:
the rule text or examples probably need rewording to match how biased sentences are written.

In [ ]:
seen = set()
for mode in ("flagged", "sentence"):
    _, r = evaluate(tests, mode, config.RAG_K)
    seen.update(i for got in r["retrieved"] for i in got)
never = [i for i in ids if i not in seen and i != config.ALWAYS_INCLUDED_RULE]
print("never retrieved:", never or "none")